In [ ]:
"""
Objetivo: Utilizar o algoritmo PELT (Pruned Exact Linear Time) para detecção 
endógena de múltiplas quebras estruturais (mudanças de regime na média e variância) 
nas taxas de inadimplência do mercado de crédito dos EUA.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ruptures as rpt
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. CARREGAMENTO DOS DADOS
# ==========================================
def carregar_dados_pelt(arquivo_csv: str) -> pd.DataFrame:
    """Carrega a base com janela temporal máxima para captura do histórico completo de quebras."""
    print("[*] Carregando a base de dados limpa...")
    df = pd.read_csv(arquivo_csv, index_col=0, parse_dates=True)
    df_model = df.loc['2000-01-01':'2026-04-01'].copy()
    return df_model

CARTEIRAS = {
    'inad_hipotecas': 'Hipotecas',
    'inad_cartoes': 'Cartões de Crédito',
    'inad_consumidor': 'Crédito ao Consumidor'
}

# ==========================================
# 2. DETECÇÃO ALGORÍTMICA (PELT) E VISUALIZAÇÃO
# ==========================================
def executar_pelt_e_plotar(df: pd.DataFrame, dicionario_carteiras: dict, arquivo_saida: str):
    """
    Roda o estimador PELT com penalidade ótima para evitar overfitting (falsas crises)
    e plota o histórico indicando as mudanças estruturais estimadas.
    """
    print("[*] Aplicando algoritmo PELT para detecção endógena de quebras...")
    
    fig, axes = plt.subplots(len(dicionario_carteiras), 1, figsize=(12, 12), sharex=True)
    
    for idx, (col, nome) in enumerate(dicionario_carteiras.items()):
        serie = df[col].dropna()
        sinal = serie.values.reshape(-1, 1)
        
        # Algoritmo PELT (L2 cost para detecção de média/variância)
        algo = rpt.Pelt(model="l2", min_size=4).fit(sinal)
        
        # Penalidade tipo BIC para evitar overfitting (dependente da variância intrínseca e tamanho)
        penalidade = np.log(len(sinal)) * sinal.var()
        quebras_idx = algo.predict(pen=penalidade)
        
        # O algoritmo retorna o índice final como quebra por padrão; isola apenas rupturas reais.
        datas_quebra = [serie.index[i-1] for i in quebras_idx if i < len(serie)]
        
        print(f" > {nome}: {len(datas_quebra)} quebra(s) estrutural(is) detectada(s).")
        
        # Plotagem da Série Temporal
        ax = axes[idx]
        ax.plot(serie.index, serie, color='#1f77b4', linewidth=2, label=f'Taxa de Inadimplência ({nome})')
        
        # Linhas e Labels de Ruptura (Mudança de Regime)
        for i, data_q in enumerate(datas_quebra):
            label_q = 'Quebra Estrutural (Mudança de Regime)' if i == 0 else ""
            ax.axvline(x=data_q, color='red', linestyle='--', linewidth=1.5, label=label_q)
            
            fator_altura = 0.95 if i % 2 == 0 else 0.85
            altura_y = ax.get_ylim()[1] * fator_altura
            
            # Formatação PT-BR (1ºTrim2008)
            ax.text(data_q, altura_y, f'  {data_q.quarter}ºTrim{data_q.year}', 
                    color='red', fontweight='bold', fontsize=9, rotation=0, va='top', ha='left')
        
        ax.set_ylabel('Inadimplência (%)', fontsize=10, fontweight='bold')
        ax.grid(True, linestyle='--', alpha=0.3)
        ax.legend(loc='upper left', frameon=False, fontsize=9)

    axes[-1].set_xlabel('Trimestres', fontsize=11, fontweight='bold')
    plt.tight_layout()
    
    plt.savefig(arquivo_saida, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\n[SUCESSO] Gráfico de quebras estruturais exportado como '{arquivo_saida}'.")

if __name__ == "__main__":
    df_model = carregar_dados_pelt('base_macro_fred_limpa.csv')
    executar_pelt_e_plotar(df_model, CARTEIRAS, '05_quebras_estruturais_credito.png')